# Stage 11 - the reranker under an RL objective

Continues the Stage 8 reranker for one epoch on new groups under two losses:
listwise cross-entropy and a policy gradient over sampled rankings.
Details: `docs/stage11_reranker_rl.md`.

## Setup

In [ ]:
# project folder on Drive
PROJECT_DIR = '/content/drive/MyDrive/RAG chunk optimize'

from google.colab import drive
drive.mount('/content/drive')

import os, sys
if not os.path.isfile(os.path.join(PROJECT_DIR, 'config.py')):
    raise RuntimeError(f'no config.py under {PROJECT_DIR!r} - fix PROJECT_DIR above.')

os.environ['RAG_DATA_ROOT'] = PROJECT_DIR + '/artifacts'
sys.path.insert(0, PROJECT_DIR)
os.chdir(PROJECT_DIR)

import config as C
C.ensure_dirs()
print('project:', PROJECT_DIR)
print(C.summary())

## Install dependencies

In [ ]:
!pip install -q -r requirements.txt

## Training groups

One positive and seven hard negatives per question, mined with the Stage 8 rule and
cached on Drive.

In [ ]:
!python scripts/26_build_stage11_data.py

## Training smoke check

Two steps per arm, written to a temporary folder and deleted.

In [ ]:
!python scripts/27_train_reranker_rl.py --smoke

## Train both arms

Each arm writes a completion marker, so a rerun skips finished arms.

In [ ]:
!python scripts/27_train_reranker_rl.py

## Dev gate

Stage 8 dev bench, fixed 15/0, R@1.

In [ ]:
!python scripts/28_eval_reranker_rl.py --dev

## Gate

In [ ]:
import json, os, pathlib

latest = pathlib.Path(os.environ['RAG_DATA_ROOT']) / 'results' / 'latest'
gate_path = latest / 'stage11_dev_gate.json'
gate = json.loads(gate_path.read_text(encoding='utf-8')) if gate_path.exists() else None
GO = bool(gate) and gate['verdict'] == 'GO'
if gate is None:
    print('no dev gate result - the dev evaluation did not finish.')
else:
    print(f"dev gate: {gate['verdict']} - {gate['why']}")
    print(f"RL - CE dev R@1 {gate['rl_minus_ce_r1']:+.4f}, 95% CI "
          f"[{gate['ci95'][0]:+.4f}, {gate['ci95'][1]:+.4f}], "
          f"RL live fraction {gate['live_fraction']:.3f}")
print('\nGO - the final evaluation will run.' if GO else
      '\nNO-GO - the final evaluation is skipped. Record the NO-GO as the result.')

## Only on GO: the Stage 6 bench

Fixed 15/0, four rerankers over the same pools, checkpointed.

In [ ]:
if GO:
    !python scripts/28_eval_reranker_rl.py
else:
    print('skipped: the dev gate did not pass.')

## Optional: fixed 6/0

Off by default and outside the verdict.

In [ ]:
RUN_SECONDARY = False   # fixed 6/0 is optional

if GO and RUN_SECONDARY:
    !python scripts/28_eval_reranker_rl.py --configs 15:0,6:0
else:
    print('secondary config skipped.')

## Review

In [ ]:
from IPython.display import Image, Markdown, display

summary = latest / 'stage11_summary.md'
if summary.exists():
    display(Markdown(summary.read_text(encoding='utf-8')))
    png = latest / 'stage11_delta.png'
    if png.exists():
        display(Image(str(png)))
else:
    print('no Stage 11 summary - the final evaluation has not run.')

## Verdicts

`INVALID`, `FAILED-OPTIMISATION`, `RL-BETTER`, `CE-BETTER` and `TIE` are defined in
`docs/stage11_reranker_rl.md`. Archiving is manual: `results/latest/` also holds
other stages' files.